In [1]:
import numpy as np
import SimpleITK as sitk
from typing import Tuple
import skimage as sk
from PIL import Image
import napari
import matplotlib.pyplot as plt
import scipy as sc
from glob import glob
#from Get_Orientation import get_cardinal_points, detect_cardinal_point, compute_transform
import pandas as pd
import os

In [ ]:
MAP_GREY_VALUES = {
    # Cardinal points
    "north": 200,   # replace with your actual grey values after
    "east":  110,   # greyscale conversion of your chosen colors
    # Region labels (map only)
    1: 29,   # e.g., blue   (0,0,255)   → L = 0.114*255 ≈ 29... 
    2: 76,  # e.g., green  (0,255,0)   → L value
    3: 150,   # e.g., red    (255,0,0)   → L value
    "exclude":  255,  # white hole
}

TISSUE_GREY_VALUES = {
    # Cardinal points
    "north": 91,   # replace with your actual grey values after
    "east":  75,
}
data_path = 'Pipeline_Test'
df_key = pd.read_csv('Annotation_Key.csv')

In [106]:
def get_tissue_bbox(img,name,tissue_id,data_path):
    mask = img < 255
    labels = sk.measure.label(mask)
    props = sk.measure.regionprops(labels)
    largest_obj = max(props, key=lambda p: p.area)
    largest_obj_label = largest_obj.label
    minc = min(props, key=lambda p: p.bbox[0])
    minr = min(props, key=lambda p: p.bbox[1])
    maxc = max(props, key=lambda p: p.bbox[2])
    maxr = max(props, key=lambda p: p.bbox[3])
    img_bbox = img[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    labels_bbox = labels[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    cropped_mask = (labels_bbox==largest_obj_label)*labels_bbox
    save_path_img_bbox = os.path.join(data_path,tissue_id,'cropped_image')
    save_path_cropped_mask = os.path.join(data_path,tissue_id,'cropped_mask')
    os.makedirs(save_path_img_bbox,exist_ok=True)
    os.makedirs(save_path_cropped_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img_bbox,'cropped_'+str(name)+'.png'),img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_cropped_mask,'cropped_mask_'+str(name)+'.png'),cropped_mask,check_contrast=False)
    return img_bbox, cropped_mask



In [83]:
def add_padding(img_bbox,cropped_mask,name,data_path):
    """ 
    reshape the cropped images to a square based on the largest dim using padding
    saves new arrays with added padding of a constant value: 255 for grey scale image, 0 for masks
    returns save path for later use
    """
    max_dim = max(img_bbox.shape)
    padding_y = (max_dim - img_bbox.shape[0]) // 2
    padding_x = (max_dim - img_bbox.shape[1]) // 2
    pad_img_bbox = np.pad(img_bbox,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=255)
    pad_mask_bbox = np.pad(cropped_mask,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
    save_path_img_bbox = os.path.join(data_path,tissue_id,'padded_cropped_image')
    save_path_cropped_mask = os.path.join(data_path,tissue_id,'padded_cropped_mask')
    os.makedirs(save_path_img_bbox,exist_ok=True)
    os.makedirs(save_path_cropped_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img_bbox,'padded_cropped_'+str(name)+'.png'),pad_img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_cropped_mask,'padded_cropped_mask_'+str(name)+'.png'),pad_mask_bbox,check_contrast=False)
    return pad_img_bbox,pad_mask_bbox



In [118]:
def get_map_region(data_path,map_base,map_id):
    MAP_GREY_VALUES = {
    # Cardinal points
    "north": 200,   # replace with your actual grey values after
    "east":  110,   # greyscale conversion of your chosen colors
    # Region labels (map only)
    1: 29,   # e.g., blue   (0,0,255)   → L = 0.114*255 ≈ 29... 
    2: 76,  # e.g., green  (0,255,0)   → L value
    3: 150,   # e.g., red    (255,0,0)   → L value
    }
    path = os.path.join(data_path,'Maps',map_base+'.png')
    map_arr = np.array(Image.open(path).convert('L'))
    grey_value = MAP_GREY_VALUES[map_id]
    map_region = (map_arr==grey_value)*map_arr
    return map_region

In [49]:
def detect_cardinal_point(arr, grey_value):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    
    # Create binary mask for this grey value
    mask = (arr * (arr == grey_value)).astype(int)
        
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {name}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([int(cy), int(cx)])  # return as (y,x)


In [16]:

def get_cardinal_points(img, grey_values_dict,name):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (y,x) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_values_dict[direction]
        pt = detect_cardinal_point(img, grey)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {name}")
        points[direction] = pt
        print(f"  {direction}: pixel (y,x) ({pt[0]:.1f}, {pt[1]:.1f})")
    return points


In [12]:

def orient_tissue(mask_points,moving_image):
    """
    Determine the flip and/or rotation needed to orient the moving image
    to match the map, using the N and E cardinal point locations.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → none
        north_is_up  and east_is_left → horizontal flip
        north_is_right and east_is_up  → -90 rotation
        north_is_right and east_is_down → 90 rotation
        north_is_left and east_is_up → Vertical flip + 90 rotation
        north_is_left and east_is_down → Vertical flip + -90 rotation
        north_is_down and east_is_left → vertical + horizontal flip
        north_is_down and east_is_right → vertical flip

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    moving_image: image array of moving image
    Returns
    -------
    flip matrix
    """
    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    #get moving image relative cardinal coordinates:
    north = (0, moving_image.shape[1]//2)
    south = (moving_image.shape[0]-1, moving_image.shape[1]//2)
    east = (moving_image.shape[0]//2, moving_image.shape[1]-1)
    west = (moving_image.shape[0]//2, 0)
    cardinals = {"north": north, "south": south, "east": east, "west": west}

    #which cardinal direction are the two reference points closest to?
    closest_north = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['north'][0] - cardinals[d][0], 
                    mask_points['north'][1] - cardinals[d][1]))
    closest_east = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['east'][0] - cardinals[d][0], 
                    mask_points['east'][1] - cardinals[d][1]))

    if closest_north == 'north' and closest_east == 'east':
        flip = None
        rotation = None
    elif closest_north == 'north' and closest_east == 'west':
        flip = 'horizontal'
        rotation = None
    elif closest_north == 'west' and closest_east == 'north':
        flip = None
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'north':
        flip = 'vertical'
        rotation = 'counter-clockwise'
    elif closest_north == 'west' and closest_east == 'south':
        flip = 'vertical'
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'south':
        flip = None
        rotation = 'counter-clockwise'
    elif closest_north == 'south' and closest_east == 'west':
        flip = 'both'
        rotation = None
    elif closest_north == 'south' and closest_east == 'east':
        flip = 'vertical'
        rotation = None
    else:
        print('Orientation does not match conditions, check image')
    print(f'Detected Flip: {flip}')
    print(f'Detected Rotation: {rotation}')
    return flip, rotation

In [54]:
def align_image(sitk_moving, sitk_fixed,data_path,tissue_id,name,flip=None,rotation=None):
    """ 
    Align moving image to map
    """
    FLIP_MATRICES = {
    "none":       [ 1,  0, 0,  1],
    "horizontal": [-1,  0, 0,  1],
    "vertical":   [1,  0, 0, -1],
    "both":       [-1,  0, 0, -1],
    }
    ROTATION_DICTIONARY = {
        'clockwise' : -90,
        'counter-clockwise': 90
    }
    save_path = os.path.join(data_path,tissue_id,'transformation_files',str(name)+'_'+tissue_id+'.tfm')
    os.makedirs(save_path,exist_ok=True)
    if flip != None and rotation != None:
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply flip with Affine
        flip_transform = sitk.AffineTransform(sitk_moving.GetDimension())
        flip_transform.SetMatrix(FLIP_MATRICES[flip])
        flip_transform.SetCenter(moving_center)
        #apply rotation with Euler
        angle = ROTATION_DICTIONARY[rotation]
        rads = np.deg2rad(angle)
        rotation_transform = sitk.Euler2DTransform()
        rotation_transform.SetCenter(moving_center)
        rotation_transform.SetAngle(rads)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(rotation_transform)
        composite_transform.AddTransform(flip_transform)

    elif flip == None and rotation != None:
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply rotation with Euler
        angle = ROTATION_DICTIONARY[rotation]
        rads = np.deg2rad(angle)
        rotation_transform = sitk.Euler2DTransform()
        rotation_transform.SetCenter(moving_center)
        rotation_transform.SetAngle(rads)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(rotation_transform)
    
    elif flip != None and rotation == None:
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply flip with Affine
        flip_transform = sitk.AffineTransform(sitk_moving.GetDimension())
        flip_transform.SetMatrix(FLIP_MATRICES[flip])
        flip_transform.SetCenter(moving_center)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(flip_transform)
    else:
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
    #Finetune alignment 
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMeanSquares()
    registration_method.SetInitialTransform(composite_transform,inPlace=True)
    registration_method.SetOptimizerAsLBFGS2(
        solutionAccuracy=1e-5,
        numberOfIterations=500,
        deltaConvergenceTolerance=0.01
        )
    registration_method.SetOptimizerScalesFromPhysicalShift()
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[8, 4, 2, 1])
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[4, 2, 1, 0])
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    registration_method.SetInterpolator(sitk.sitkLinear)
    #registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
    print(f"  Registering...")
    optimized_transform = registration_method.Execute(fixed_arr,moving_arr)
    sitk.WriteTransform(optimized_transform,save_path)
    print(f"Metric: {registration_method.GetMetricValue():.6f}")
    return optimized_transform

In [122]:
def calculate_spacing(img_mask,map_region_mask):
    #get area of tissue mask bounding box
    moving_labels = sk.measure.label(img_mask)
    props = sk.measure.regionprops(moving_labels)
    minc,minr,maxc,maxr = props[0].bbox
    moving_height_px = maxr - minr
    moving_width_px  = maxc - minc
    moving_area_px = moving_height_px * moving_width_px
    #get area of map region bounding box
    map_region_label = sk.measure.label(map_region_mask)
    map_region_props = sk.measure.regionprops(map_region_label)
    minc,minr,maxc,maxr = map_region_props[0].bbox
    map_height_px = maxr - minr
    map_width_px  = maxc - minc
    map_area_px = map_height_px * map_width_px
    spacing = round(np.sqrt(moving_area_px//map_area_px),1)
    map_spacing = (float(spacing),float(spacing))
    return map_spacing

In [ ]:
def load_sitk_img()

In [87]:
test_df = df_key[df_key['AnimalID'] == 'SD211']

In [88]:
test_left_df = test_df[test_df['Gland.side'] == 'Left']
test_left_df.head()

,Unnamed: 0,Image,Genotype,Timepoint,AnimalID,Gland.side,RegionalNumber,Tumor.Animal,Tissue.ID,MappingID,Max.Region,MapBase
982,983,205892,WT,IV28,SD211,Left,3,N,Tissue1,3.0,3,SD211-Left
983,984,205892,WT,IV28,SD211,Left,1,N,Tissue2,1.0,3,SD211-Left
984,985,205893,WT,IV28,SD211,Left,2,N,Tissue,2.0,3,SD211-Left


In [ ]:
flip_series = []
rotation_series = []
save_path_cropped_masks = []
padded_imgs = []
padded_masks = []

image_names = test_left_df['Image'].to_list()
tissue_ids = test_left_df['Tissue.ID'].to_list()
map_ids = test_left_df['MappingID'].to_list()
map_base = test_left_df['MapBase'].to_list()

test_name = str(image_names[0])
test_tissue_id = tissue_ids[0]
test_map_id = map_ids[0]
map = map_base[0]

img_path = os.path.join(data_path,test_tissue_id,test_name+'.svs.png')
img = np.array(Image.open(img_path).convert('L'))
img_bbox, cropped_mask = get_tissue_bbox(img,test_name,test_tissue_id,data_path)
pad_img_bbox, pad_mask_bbox = add_padding(img_bbox,cropped_mask,test_name,data_path)
points = get_cardinal_points(img_bbox,TISSUE_GREY_VALUES,test_name)
flip, rotation = orient_tissue(points,img_bbox)
flip_series.append(flip)
rotation_series.append(rotation)
save_path_cropped_masks.append(mask_save_path)
padded_imgs.append(pad_img_bbox)
padded_masks.append(pad_mask_bbox)

    

  north: pixel (y,x) (25.0, 813.0)
  east: pixel (y,x) (2364.0, 1565.0)
Detected Flip: None
Detected Rotation: None


In [111]:
viewer = napari.view_image(pad_img_bbox)

E:\Temp\ipykernel_26160\584714893.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(pad_img_bbox)


In [119]:
map_region = get_map_region(map_base=map,map_id=test_map_id,data_path=data_path)

In [123]:
map_spacing = calculate_spacing(pad_mask_bbox,map_region)

In [124]:
map_spacing

(2.0, 2.0)